<a href="https://colab.research.google.com/github/chetools/CHE4061_Spring2026/blob/main/corrected_dynamic_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [321]:
alpha = 2.
F = 1.
zF = 0.45
q=0.8  #all liquid feed
Dsp = 0.55*F
Rinit=1.5

N= 7
NFeed = 4
M0stage = 10.
M0cond = 20.
M0boil = 30.
c = 0.1  #weir constant
M0 = np.r_[M0cond, N*[M0stage], M0boil]

#correction
Minitial = np.r_[((Rinit+1)*Dsp/c)**(2/3)+M0cond, N*[15], 1.1*M0boil]
Minitial


array([25.73942606, 15.        , 15.        , 15.        , 15.        ,
       15.        , 15.        , 15.        , 33.        ])

In [322]:
c*(Minitial-M0)**1.5

array([1.375     , 1.11803399, 1.11803399, 1.11803399, 1.11803399,
       1.11803399, 1.11803399, 1.11803399, 0.51961524])

In [323]:
def ramp_factory(y1, y2, t1, t2):
    def ramp(t):
        if t<t1:
            return y1
        if t>t2:
            return y2
        return y1+(y2-y1)*(t-t1)/(t2-t1)
    return ramp

In [324]:
ramp = ramp_factory(2, 5, 10, 30)

In [329]:
def rhs(t, vec):

    R = Rinit
    M = vec

    L = c*np.where(M<M0, 0, (M-M0))**1.5
    D = L[0]/(R+1)

    Vrec = (R+1)*D
    Vstrip = Vrec - (1-q)*F

    dM = np.zeros_like(M)
    dM -= L
    dM[1:]+=L[:-1]
    dM[1]-=D

    dM[:NFeed-1]+=Vrec
    dM[NFeed:-1]+=Vstrip

    dM[1:NFeed]-=Vrec
    dM[NFeed:]-=Vstrip

    dM[NFeed]+=q*F
    dM[NFeed-1]+=(1-q)*F+Vstrip

    return dM

In [330]:
tend =200
tplot = np.linspace(0,tend, 200)
res = sp.integrate.solve_ivp(rhs, (0,tend), y0=Minitial, method='Radau', dense_output=True)
Ms = res.sol(tplot)

In [331]:
Rplot = np.array([ramp(t) for t in tplot])

In [332]:
fig=make_subplots(rows=1,cols=2)
for i, M in enumerate(Ms):
    fig.add_scatter(x=tplot, y=M, name=f'{i}', row=1, col=1)
fig.add_scatter(x=tplot, y=Rplot, row=1, col=2)
fig.update_layout(width=800, height=400)

In [333]:
def rhs2(t, vec):
    #If feed is two phase, calculate the mole fraction of more volatile component in both liquid and vapor phase (xF,yF)
    #zF = x*q + alpha*x /(1-x+alpha*x) * (1-q)
    aa= alpha*q-q
    bb = alpha - alpha*q + zF - alpha*zF + q
    cc = -zF
    xF = (-bb + np.sqrt(bb**2-4*aa*cc))/(2*aa) #, (-bb - np.sqrt(bb**2-4*aa*cc))/(2*aa)
    yF = alpha*xF/(1-xF+alpha*xF)

    # R = ramp(t)
    R=Rinit
    x, M = np.split(vec,2)
    y = alpha*x/(1-x+alpha*x)

    L = c*np.where(M<M0, 0, (M-M0))**1.5
    D = L[0]/(R+1)

    Vrec = (R+1)*D
    Vstrip = Vrec - (1-q)*F

    dM = np.zeros_like(M)
    dM -= L
    dM[1:]+=L[:-1]
    dM[1]-=D

    dM[:NFeed]+=Vrec
    dM[NFeed:-1]+=Vstrip

    dM[1:NFeed]-=Vrec
    dM[NFeed:]-=Vstrip

    dM[NFeed]+=q*F
    dM[NFeed-1]+=(1-q)*F

    dxM = np.zeros_like(M)
    dxM -= x*L
    dxM[1:]+=x[:-1]*L[:-1]
    dxM[1]-=x[0]*D

    dxM[:NFeed-1]+=y[1:NFeed]*Vrec
    dxM[NFeed:-1]+=y[NFeed+1:]*Vstrip

    dxM[1:NFeed]-=y[1:NFeed]*Vrec
    dxM[NFeed:]-=y[NFeed:]*Vstrip

    dxM[NFeed]+=xF*q*F
    dxM[NFeed-1]+=yF*(1-q)*F+y[NFeed]*Vstrip

    #dxM = x dM + M dx  Product role
    dx = (dxM - x*dM)/M

    return np.r_[dx, dM]

In [334]:
tend =5000
tplot = np.linspace(0,tend, 200)
res = sp.integrate.solve_ivp(rhs2, (0,tend), y0=np.r_[x_initial, Minitial], method='Radau', dense_output=True)
xs, Ms = np.split(res.sol(tplot),2, axis=0)

In [335]:
fig=make_subplots(rows=1,cols=2)
for i, (x,M) in enumerate(zip(xs,Ms)):
    fig.add_scatter(x=tplot, y=x, name=f'x{i}', row=1, col=1)
    fig.add_scatter(x=tplot, y=M, name=f'M{i}', row=1, col=2)

fig.update_layout(width=1200, height=400)